# **SAGE: Practice**

Hi, everyone. In the lesson “SHAP for other things: interactions and globality” we moved Shapley sharing from a single prediction to the quality of the model as a whole and arrived at SAGE (Shapley Additive Global importancE, [Covert, Lundberg & Lee, 2020](https://arxiv.org/abs/2004.00668)). The players in this game are the features, and the value of a coalition $S$ is how much knowing the features in $S$ reduces the error:

$$v(S) = E\big[\ell(f_\varnothing, Y)\big] - E_{X,Y}\big[\ell(f_S(X), Y)\big],$$

where $f_S$ is the model's prediction with the missing features marginalized over a background sample, and $f_\varnothing = E[f(X)]$ is the “empty” model that always outputs the model's mean prediction. The importance $\phi_i$ of a feature is its Shapley value in the game $(N, v)$, and by efficiency $\sum_i \phi_i = v(N)$, the total reduction in error.

Today we compute SAGE with the [`sage-importance`](https://github.com/iancovert/sage) library on the familiar diabetes dataset: in the same lesson we built the mean|φ| ranking on it.

In this practice you will:
- compute SAGE values for a random forest with `sage.MarginalImputer` and `sage.PermutationEstimator`;
- check efficiency: add up the SAGE values and compare the sum with $v(N)$ computed by hand;
- compare SAGE with permutation importance from `sklearn`;
- add a near-exact copy of a feature to the data and see how SAGE and permutation importance split the contribution between the duplicates.

**Happy coding!**

As always, first install the dependencies. Only `sage-importance` is missing on Colab; we pin its version so that the numbers match ours.

In [ ]:
!pip install sage-importance==0.0.6 -q

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sage

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, r2_score

## Data and model

The diabetes dataset ships with `sklearn`, so nothing has to be downloaded: 442 patients, 10 features (age, sex, body mass index `bmi`, blood pressure `bp` and six blood measurements `s1`–`s6`), and the target is a quantitative measure of disease progression one year later. The features are already centered and scaled. We hold out 30% of the rows as a test set: both SAGE and permutation importance will be computed on it.

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, test_size=0.3, random_state=0)
print(X.shape, X_train.shape, X_test.shape)
X.head()

The model is a random forest with 300 trees. The parameter `max_features=0.33` means that at every split a tree picks the best feature from a random third of the features. For the first sections this is only regularization, but in the duplicate experiment this property matters: the forest will use both copies of the feature.

In [ ]:
def fit_forest(X_train, y_train):
    return RandomForestRegressor(n_estimators=300, max_features=0.33, min_samples_leaf=3,
                                 random_state=0, n_jobs=-1).fit(X_train, y_train)

model = fit_forest(X_train, y_train)
pred_test = model.predict(X_test)
print(f'Test MSE: {mean_squared_error(y_test, pred_test):.1f}, test R^2: {r2_score(y_test, pred_test):.3f}')

## SAGE values

In `sage-importance` the computation is assembled from two parts.

- **`sage.MarginalImputer(model, background)`** implements $f_S$: the missing features are replaced with the values from each background row, and the predictions are averaged. The background is 64 random rows of the training set; a larger background gives a more accurate marginalization and a slower computation.
- **`sage.PermutationEstimator(imputer, 'mse')`** estimates Shapley values by Monte Carlo. For one random test row it takes a random permutation of the features, adds the features one at a time and records how much the squared error drops after each feature is added. The average of these increments over many permutations is the estimate of $\phi_i$. There is also `sage.KernelEstimator`, which solves the same problem through weighted regression, like KernelSHAP.

By default the estimator stops by itself once the confidence intervals are narrow enough, and then the number of permutations varies from run to run. We fix the number of permutations (8192), the estimator's seed and the background so that a rerun gives the same numbers. The computation takes about a minute.

In [ ]:
N_PERMUTATIONS = 8192

def sage_values(model, X_train, X_test, y_test):
    random.seed(0)
    np.random.seed(0)
    background = X_train[np.random.default_rng(0).choice(len(X_train), 64, replace=False)]
    imputer = sage.MarginalImputer(model, background)                        # f_S: marginalization over the background
    estimator = sage.PermutationEstimator(imputer, 'mse', random_state=0)    # Shapley values via permutations
    explanation = estimator(X_test, y_test, batch_size=512, n_permutations=N_PERMUTATIONS,
                            detect_convergence=False, bar=False)
    return explanation, background

sage_expl, background = sage_values(model, X_train, X_test, y_test)
# std is the standard error of the Monte Carlo estimate, not the spread across objects
pd.DataFrame({'SAGE': sage_expl.values, 'std': sage_expl.std}, index=feature_names).sort_values('SAGE', ascending=False).round(1)

SAGE values are measured in the same units as the loss: $\phi_i$ is the part of the reduction in test MSE assigned to feature $i$. The library has a built-in plot with 95% confidence intervals of the estimate.

In [ ]:
sage_expl.plot(feature_names, title='SAGE values, random forest on diabetes', figsize=(9, 5))

**Self-check question.** Which two features received the largest SAGE values? Does this estimate distinguish them once the 95% confidence intervals are taken into account? For features with negative SAGE values, what does the minus mean in terms of error?

<details><summary>Check yourself</summary>

The leaders are `bmi` (648.7, standard error 30.7) and `s5` (639.8, standard error 28.8). The gap between them is 8.9, while the half-width of the 95% confidence interval of each is about 60 (1.96 standard errors), so the intervals overlap: this estimate does not distinguish `bmi` from `s5`, and their order may flip under another estimator seed. All we can say for sure is that both features matter more than `bp` (248.6). Negative values belong to `age` (−11.4) and `s1` (−9.8): averaged over coalitions, knowing these features increases the squared error on the test set, so the model has probably fitted noise in these features on the training set.

</details>

## Task 1. Efficiency: the sum of SAGE values and $v(N)$

By the theory $\sum_i \phi_i = v(N)$, where $v(N)$ is the reduction in error achieved by the model with all features relative to an “empty” model. Compute $v(N)$ on the test set by hand in two ways:

1. the “empty” model predicts $\bar y$, the mean target on the training set: the simplest way to get a baseline error;
2. as in the definition from the theory and the way `MarginalImputer` does it: for the empty coalition every feature comes from the background, so the “empty” prediction $f_\varnothing$ is the model's mean prediction over the rows of `background`.

Compare both quantities with the sum of the SAGE values.

A hint on why the match is not exact. Within one permutation the error increments add up to the difference between the losses of the empty and the full model on one row (the intermediate terms cancel). So the sum of the estimates equals the mean of this difference over the 8192 rows that the estimator drew from the test set with replacement. Compute the standard error of that mean: the standard deviation of the difference over the test rows divided by $\sqrt{8192}$.

In [ ]:
loss_full = # Your code here: mean squared error of the full model on the test set

v_N_ybar = # Your code here
f_empty = # Your code here
v_N_empty = # Your code here

standard_error = # Your code here

print(f'Sum of SAGE values:                 {sage_expl.values.sum():.1f}')
print(f'v(N) with empty model y_bar:        {v_N_ybar:.1f}')
print(f'v(N) with empty model f_empty:      {v_N_empty:.1f}')
print(f'Sum minus v(N) with f_empty:        {sage_expl.values.sum() - v_N_empty:.1f}, standard error: {standard_error:.1f}')

**Self-check question.** What are the sum of the SAGE values and $v(N)$ on the test set for the two choices of the “empty” model? Is the difference within two standard errors?

<details><summary>Check yourself</summary>

The sum of the SAGE values is 1790.7. $v(N)$ with the empty model $\bar y = 152.1$ is 1767.6, and with the empty model $f_\varnothing = 148.4$ (the mean prediction over the background) it is 1781.8. The difference between the sum and $v(N)$ with $f_\varnothing$ is 8.9 with a standard error of 69.0, i.e. within one standard error: efficiency holds up to the noise of row sampling. The right comparison is with $f_\varnothing$: the library measures the error reduction from the mean prediction over the background, not from the mean target.

</details>

## Task 2. Comparison with permutation importance

Permutation importance shuffles one feature while keeping all the others in place and measures how much the error grows. In game terms this is the marginal contribution of the feature to the coalition of all other features, that is, a single term of the Shapley sum instead of an average over all coalitions.

Compute the permutation importance of the random forest on the test set with `sklearn.inspection.permutation_importance`, using `scoring='neg_mean_squared_error'` (so that the importance is measured in the same MSE units as SAGE), `n_repeats=10` and `random_state=0`. Store the result in `perm`, build a table `comparison` with columns `SAGE` and `permutation`, and compare the column sums with $v(N)$.

In [ ]:
random.seed(0)
np.random.seed(0)
perm = # Your code here
comparison = # Your code here
print(comparison.round(1))
print(f'\nSum of SAGE: {comparison.SAGE.sum():.1f}, sum of permutation importance: {comparison.permutation.sum():.1f}, v(N) with f_empty: {v_N_empty:.1f}')

Let us plot the two importances side by side. They share units, the reduction in test MSE, so the bars can be compared directly.

In [ ]:
def plot_two(table, title):
    pos = np.arange(len(table))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(pos - 0.2, table.SAGE, height=0.4, label='SAGE', color='tab:green')
    ax.barh(pos + 0.2, table.permutation, height=0.4, label='permutation importance', color='tab:gray')
    ax.set_yticks(pos, table.index)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('reduction in test MSE')
    ax.set_title(title)
    ax.legend()
    plt.show()

plot_two(comparison, 'SAGE and permutation importance, random forest on diabetes')

**Self-check question.** Do the top two positions of the ranking coincide for SAGE and for permutation importance? What is the sum of permutation importance over all features, and why does it not have to equal $v(N)$?

<details><summary>Check yourself</summary>

The top two positions coincide: `bmi` and `s5` for both methods (permutation importance 743.9 and 734.1). The sum of permutation importance is 1663.1 against $v(N) = 1781.8$: permutation importance is not a Shapley value, it has no efficiency property, and its sum does not have to give anything in particular. The disagreements are concentrated in the blood measurements: `s4` and `s3` have positive SAGE values (97.4 and 74.6) but negative permutation importance (−27.9 and −42.8). These features are correlated with others (`s3` and `s4`: −0.74, `s4` and `s2`: 0.66), and when one is shuffled the others stay in place and make up for the information, so permutation importance sees only this last marginal contribution.

</details>

## Task 3. A redundant feature: how SAGE and permutation importance split the contribution

Now let us check the main promise of the theory: SAGE handles redundant features more correctly. We add a noisy copy of `s5` to the data: `s5_dup = s5 + noise`, where the noise standard deviation is 10% of the standard deviation of `s5`. `s5_dup` carries no new information about the target. We train the same forest on 11 features: with `max_features=0.33`, in some splits `s5` is not among the random third of the features while `s5_dup` is, so the forest uses both copies.

In [ ]:
X_dup = X.copy()
X_dup['s5_dup'] = X['s5'] + np.random.default_rng(0).normal(0, 0.1 * X['s5'].std(), len(X))
feature_names_dup = list(X_dup.columns)

X_dup_train, X_dup_test, _, _ = train_test_split(X_dup.values, y.values, test_size=0.3, random_state=0)
model_dup = fit_forest(X_dup_train, y_train)
print(f"Correlation of s5 and s5_dup: {np.corrcoef(X_dup['s5'], X_dup['s5_dup'])[0, 1]:.3f}")
print(f'Test MSE: without the duplicate {mean_squared_error(y_test, model.predict(X_test)):.1f}, with the duplicate {mean_squared_error(y_test, model_dup.predict(X_dup_test)):.1f}')

Compute SAGE values for `model_dup` (with the `sage_values` function above) and permutation importance (with the same parameters as in Task 2). Build a table `comparison_dup` with columns `SAGE` and `permutation` over the 11 features and compute the share `dup_share`, the fraction of the combined SAGE value of the pair `s5` and `s5_dup` that went to the duplicate.

In [ ]:
sage_dup, _ = # Your code here
random.seed(0)
np.random.seed(0)
perm_dup = # Your code here
comparison_dup = # Your code here
dup_share = # Your code here

pair = pd.DataFrame({
    'SAGE without dup': [comparison.SAGE['s5'], np.nan, comparison.SAGE['s5']],
    'SAGE with dup': [comparison_dup.SAGE['s5'], comparison_dup.SAGE['s5_dup'], comparison_dup.SAGE[['s5', 's5_dup']].sum()],
    'permutation without dup': [comparison.permutation['s5'], np.nan, comparison.permutation['s5']],
    'permutation with dup': [comparison_dup.permutation['s5'], comparison_dup.permutation['s5_dup'], comparison_dup.permutation[['s5', 's5_dup']].sum()],
}, index=['s5', 's5_dup', 's5 + s5_dup'])
print(pair.round(1).to_string())
print(f'\nDuplicate share of the pair SAGE: {dup_share:.3f}')
print(f'Sum of SAGE: {comparison_dup.SAGE.sum():.1f}, standard error of s5 and s5_dup: {sage_dup.std[feature_names_dup.index("s5")]:.1f} and {sage_dup.std[-1]:.1f}')

Let us look at both importances of the model with the duplicate.

In [ ]:
plot_two(comparison_dup, 'Model with the duplicate s5_dup: SAGE and permutation importance')

**Self-check question.** What share of the SAGE value of the pair `s5` + `s5_dup` went to the duplicate? Compare the pair's combined SAGE value with the SAGE value of `s5` in the model without the duplicate. By what factor did the permutation importance of `s5` drop after the duplicate was added, and what is the permutation importance of the pair?

<details><summary>Check yourself</summary>

The duplicate received a share of 0.489 of the pair's SAGE value: `s5` 357.2, `s5_dup` 342.5, together 699.7, close to the 639.8 that `s5` received alone in the model without the duplicate (a difference of 59.9 with standard errors of 20.1 and 18.8 for the two terms, and this is a different trained model). SAGE split the contribution between the copies almost evenly and preserved their combined contribution. The permutation importance of `s5` fell from 734.1 to 59.9, about 12 times, `s5_dup` gets 37.8, and the pair together 97.7: when one copy is shuffled the other stays in place, the forest keeps predicting from it, and both copies look nearly useless.

</details>

## Conclusion

Today you:
- computed SAGE values for a random forest with the `sage-importance` library and saw that the Monte Carlo estimate has a standard error that must be taken into account when features are compared;
- checked efficiency: the sum of SAGE values estimates $v(N)$, the total reduction in error relative to an “empty” model, and it matters which prediction counts as “empty”;
- compared SAGE with permutation importance: the two leaders coincide, for the correlated blood measurements `s3` and `s4` the two disagree even in sign, and the sum of permutation importance does not have to give $v(N)$;
- saw the main difference on a redundant feature: SAGE splits the contribution between the duplicates and preserves their combined contribution, while permutation importance devalues both copies because it evaluates each one with the other still in place.

See you in the next steps!